### Importing the libraries

In [ ]:
import numpy as np
import pandas as pd
import torch as t

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn   

from torchinfo import summary

### Loading & Cleaning the Dataset

In [ ]:
df = pd.read_csv(r'ML Practice\Breast_Cancer_Dataset.csv')
print("Dataset Loaded Successfully")

In [ ]:
print(df.head())

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
if 'Unnamed: 32' in df.columns:
    df = df.drop(columns=['Unnamed: 32'])
    

In [ ]:
df['diagnosis'] = df['diagnosis'].map({
    'M': 1,
    'B': 0
})

In [ ]:
df['diagnosis'].info()

### Basic Preprocessing

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    df.iloc[:, 2:], # Skipping 'id' (col 0) and 'diagnosis' (col 1) to only get math features
    df['diagnosis'],
    test_size=0.2,
    random_state=42 
)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
import torch
print(torch.version.cuda)

In [ ]:
X_train_tensor = t.as_tensor(X_train, dtype=t.float32).to('cuda')
X_test_tensor = t.as_tensor(X_test, dtype=t.float32).to('cuda')

# Force extraction of the raw numbers and reshape to a 2D column (455, 1)
Y_train_tensor = t.as_tensor(np.array(Y_train), dtype=t.float32).view(-1, 1).to('cuda')
Y_test_tensor = t.as_tensor(np.array(Y_test), dtype=t.float32).view(-1, 1).to('cuda')

print(f"X Train Shape: {X_train_tensor.shape} Y Train Shape: {Y_train_tensor.shape}")
print(f"X Test Shape: {X_test_tensor.shape} Y Test Shape: {Y_test_tensor.shape}")

### Training process

1) Create the model
2) Forward pass
3) Loss function 
4) Backward pass
5) Parameter update


In [ ]:
class My_NN():
    def __init__(self, num_features):
        # Initialize weights based on how many features we have.
        # Using float32 to keep the math lightweight and fast.
        self.weights = t.randn(num_features, 1, dtype=t.float32, requires_grad=True)
        self.bias = t.zeros(1, dtype=t.float32, requires_grad=True)

    def forward(self, X):
        # The core math equation: (X * W) + b
        z = t.matmul(X, self.weights) + self.bias
        # Squish the result between 0 and 1 to get a percentage/probability
        y_pred = t.sigmoid(z)
        return y_pred
    
    def binary_cross_entropy(self, y_pred, y):
        # The Float32 Precision Trap Fix
        # 1e-7 is too microscopic for a 32-bit hardware architecture to "see". 
        # We use 1e-5 so the computer registers the boundary limit, preventing log(0) crashes.
        epsilon = 1e-5
        y_pred = t.clamp(y_pred, epsilon, 1 - epsilon)

        # The Loss Equation: How wrong was the model?
        loss = -(y * t.log(y_pred) + (1 - y) * t.log(1 - y_pred)).mean()
        return loss

In [ ]:
lr = 0.1
epochs = 50

# Pass the number of features dynamically (shape[1] = number of columns)
model = My_NN(X_train_tensor.shape[1])

for epoch in range(epochs):
    # Step 1: Forward Pass (Using the TENSORS, not the NumPy arrays!)
    y_pred = model.forward(X_train_tensor)
    
    # Calculate Accuracy (First-Principles: If probability >= 50%, call it class 1 (Malignant), else 0 (Benign))
    predictions_class = (y_pred >= 0.5).float()
    accuracy = (predictions_class == Y_train_tensor).float().mean() * 100
    
    # Step 2: Loss Calculation
    loss = model.binary_cross_entropy(y_pred, Y_train_tensor)
    
    # Step 3: Backward Pass (Calculate Gradients/Derivatives)
    loss.backward()

    # Step 4: Parameter Update (Learning)
    with t.no_grad(): # Turn off tape recorder so we can change the weights
        model.weights -= lr * model.weights.grad
        model.bias -= lr * model.bias.grad
    
    # Step 5: Wipe the chalk board! (Zero Gradients)
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    print(f"--- Epoch {epoch+1}/{epochs} ---")
    print(f"Loss: {loss.item():.4f} | Accuracy: {accuracy.item():.2f}%")
     # Show a peek of the exact calculations for the first 3 patients
    print(f"Raw Probabilities: {y_pred[:3].detach().numpy().flatten().round(3)}")
    print(f"Final Prediction:  {predictions_class[:3].detach().numpy().flatten()}")
    print(f"Actual Truth:      {Y_train_tensor[:3].detach().numpy().flatten()}\n")

print("\nTraining Complete. Model has learned the patterns!")

In [ ]:
import torch
import torch.nn as nn   

In [ ]:
class Model(nn.Module):
    def __init__(self, num_features):

        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 3),
            nn.ReLU(),
            nn.Linear(3,1),
            nn.Sigmoid()
        )

    def forward(self, features):
        out = self.network(features)
        
        return out
    
    